In [ ]:
"""
Data Module - Fetches stock prices from Yahoo Finance
Simple wrapper that gets 2 years of price history
"""

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


class DataFetcher:
    """Gets stock data and calculates basic statistics"""
    
    def __init__(self, ticker, peers):
        """
        Args:
            ticker: Your target stock (e.g., 'OKLO')
            peers: List of comparison stocks (e.g., ['SMR', 'BWXT', 'CCJ'])
        """
        self.ticker = ticker
        self.peers = peers
        self.all_tickers = [ticker] + peers
        self.prices = None
        self.returns = None
        self.current_prices = {}
    
    def fetch(self):
        """Download 2 years of price data"""
        print(f"\n Fetching data for {self.ticker} and {len(self.peers)} peers...")
        
        # FIX: yfinance now defaults to auto_adjust=True, so use 'Close' instead of 'Adj Close'
        data = yf.download(self.all_tickers, period='2y', progress=False)['Close']
        
        if len(self.all_tickers) == 1:
            data = data.to_frame(name=self.all_tickers[0])
        
        # FIX: fillna with method='ffill' is deprecated, use ffill() directly
        data = data.ffill().dropna()
        self.prices = data
        
        for t in self.all_tickers:
            self.current_prices[t] = float(data[t].iloc[-1])
        
        print(f"   ✓ Got {len(data)} trading days")
        print(f"   ✓ {self.ticker} current price: ${self.current_prices[self.ticker]:.2f}")
        
        return data
    
    def calculate_statistics(self):
        """Calculate returns and volatility (annualized)"""
        daily_returns = self.prices.pct_change().dropna()
        
        self.mean_returns = daily_returns.mean() * 252
        self.cov_matrix = daily_returns.cov() * 252
        self.returns = daily_returns
        
        self.volatility = np.sqrt(np.diag(self.cov_matrix))
        
        return self.mean_returns, self.cov_matrix


if __name__ == "__main__":
    fetcher = DataFetcher('OKLO', ['SMR', 'BWXT', 'CCJ'])
    fetcher.fetch()
    fetcher.calculate_statistics()
    print("Test successful!")

In [ ]:
"""
Peer Analysis Module - Shows which stocks move together
Uses hierarchical clustering to group similar stocks
"""

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import seaborn as sns


class PeerAnalysis:
    """Analyzes how your stock compares to peers"""
    
    def __init__(self, ticker, tickers, cov_matrix):
        """
        Args:
            ticker: Your target stock
            tickers: All stocks including target
            cov_matrix: How stocks move together (from DataFetcher)
        """
        self.ticker = ticker
        self.tickers = tickers
        self.cov_matrix = cov_matrix
        self.correlation = None
    
    def analyze(self):
        """Run the peer comparison analysis"""
        print(f"\n🔍 Analyzing {self.ticker} vs peers...")
        
        # Convert cov_matrix to numpy array if it's a DataFrame
        cov_values = self.cov_matrix.values if hasattr(self.cov_matrix, 'values') else self.cov_matrix
        
        std = np.sqrt(np.diag(cov_values))
        self.correlation = cov_values / np.outer(std, std)
        
        n = len(self.tickers)
        distance = np.zeros((n, n))
        
        for i in range(n):
            for j in range(n):
                distance[i, j] = np.sqrt(
                    cov_values[i, i] +
                    cov_values[j, j] -
                    2 * cov_values[i, j]
                )
        
        condensed = squareform(distance)
        self.linkage_result = linkage(condensed, method='single')
        
        n_clusters = min(3, len(self.tickers))
        labels = fcluster(self.linkage_result, n_clusters, criterion='maxclust')
        
        results = []
        for i, ticker in enumerate(self.tickers):
            same_cluster = [self.tickers[j] for j, l in enumerate(labels) if l == labels[i] and j != i]
            
            if same_cluster:
                avg_corr = np.mean([self.correlation[i, self.tickers.index(t)] for t in same_cluster])
            else:
                avg_corr = 1.0
            
            if ticker in ['OKLO', 'SMR']:
                cluster_name = 'SMR Leaders'
            elif ticker in ['BWXT']:
                cluster_name = 'Nuclear Support'
            elif ticker in ['CCJ']:
                cluster_name = 'Uranium'
            else:
                cluster_name = f'Group {labels[i]}'
            
            results.append({
                'Ticker': ticker,
                'Cluster': cluster_name,
                'Correlation': round(avg_corr, 2)
            })
        
        self.results_table = pd.DataFrame(results)
        
        print("   ✓ Peer clusters:")
        print(self.results_table.to_string(index=False))
        
        return self.results_table
    
    def plot_similarity_map(self, output_path='peer_similarity_map.png'):
        """Create the visual tree showing stock relationships"""
        plt.figure(figsize=(10, 6))
        
        dendrogram(
            self.linkage_result,
            labels=self.tickers,
            leaf_font_size=12,
            color_threshold=0.7 * max(self.linkage_result[:, 2])
        )
        
        plt.title(f'{self.ticker} vs Nuclear Peers - Similarity Map',
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel('Stock', fontsize=12)
        plt.ylabel('Distance (lower = more similar)', fontsize=12)
        
        plt.text(
            0.5, -0.15,
            'This tree shows which stocks move together. Stocks grouped close are similar.',
            ha='center', transform=plt.gca().transAxes, fontsize=10, style='italic'
        )
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"   ✓ Saved similarity map: {output_path}")
        return output_path


if __name__ == "__main__":
    print("Peer Analysis Module Ready")

In [ ]:
"""
Price Target Module - Simulates future prices
Runs 10,000 scenarios to find realistic targets
"""

import numpy as np
import matplotlib.pyplot as plt


class PriceTargets:
    """Calculates base, upside, and downside price targets"""
    
    def __init__(self, ticker, current_price, expected_return, volatility, cov_matrix, all_returns):
        """
        Args:
            ticker: Your target stock
            current_price: Today's price
            expected_return: Expected annual return (adjusted for AI/ESG)
            volatility: Annual volatility
            cov_matrix: Covariance matrix for all stocks
            all_returns: Expected returns for all stocks (vector)
        """
        self.ticker = ticker
        self.current_price = current_price
        self.expected_return = expected_return
        self.volatility = volatility
        self.cov_matrix = cov_matrix
        self.all_returns = all_returns
    
    def simulate(self, n_sims=10000, horizon=1.0, seed=42):
        """
        Run Monte Carlo simulation
        
        Returns base (50th percentile), upside (75th), downside (25th) targets
        """
        print(f"\n Running {n_sims:,} price simulations for {self.ticker}...")
        
        np.random.seed(seed)
        
        # Convert to numpy arrays if needed
        all_returns_arr = self.all_returns.values if hasattr(self.all_returns, 'values') else self.all_returns
        cov_matrix_arr = self.cov_matrix.values if hasattr(self.cov_matrix, 'values') else self.cov_matrix
        
        random_returns = np.random.multivariate_normal(
            all_returns_arr * horizon,
            cov_matrix_arr * horizon,
            n_sims
        )
        
        ticker_returns = random_returns[:, 0]
        simulated_prices = self.current_price * np.exp(ticker_returns)
        
        self.downside = np.percentile(simulated_prices, 25)
        self.base = np.percentile(simulated_prices, 50)
        self.upside = np.percentile(simulated_prices, 75)
        self.simulated_prices = simulated_prices
        
        upside_pct = (self.base / self.current_price - 1) * 100
        
        print(f"   ✓ Current: ${self.current_price:.2f}")
        print(f"   ✓ Base Target (50th %ile): ${self.base:.2f} (+{upside_pct:.1f}%)")
        print(f"   ✓ Upside Target (75th %ile): ${self.upside:.2f}")
        print(f"   ✓ Downside (25th %ile): ${self.downside:.2f}")
        
        return {
            'current': self.current_price,
            'downside': self.downside,
            'base': self.base,
            'upside': self.upside,
            'upside_pct': upside_pct
        }
    
    def probability_of_target(self, target_price):
        """Calculate chance of reaching a specific price"""
        prob = np.mean(self.simulated_prices >= target_price)
        return prob
    
    def plot_distribution(self, output_path='price_targets.png', analyst_targets=None):
        """Create histogram showing all possible outcomes"""
        fig, ax = plt.subplots(figsize=(12, 7))
        
        ax.hist(self.simulated_prices, bins=50, alpha=0.7, color='skyblue',
                edgecolor='black', label='Simulated Outcomes')
        
        ax.axvline(self.current_price, color='red', linestyle='--', linewidth=2.5,
                   label=f'Current: ${self.current_price:.2f}')
        ax.axvline(self.downside, color='orange', linestyle='--', linewidth=2,
                   label=f'Downside (25th): ${self.downside:.2f}')
        ax.axvline(self.base, color='green', linestyle='-', linewidth=3,
                   label=f'Base Target (50th): ${self.base:.2f}')
        ax.axvline(self.upside, color='purple', linestyle='--', linewidth=2,
                   label=f'Upside (75th): ${self.upside:.2f}')
        
        if analyst_targets:
            for target in analyst_targets:
                prob = self.probability_of_target(target)
                ax.axvline(target, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)
                ax.text(
                    target, ax.get_ylim()[1]*0.85,
                    f'${target}\n{prob:.0%} chance',
                    ha='center', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
                )
        
        ax.set_title(f'{self.ticker} 1-Year Price Distribution (10,000 Scenarios)',
                     fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Price ($)', fontsize=12)
        ax.set_ylabel('Number of Scenarios', fontsize=12)
        ax.legend(fontsize=10, loc='upper right')
        ax.grid(alpha=0.3)
        
        fig.text(
            0.5, -0.02,
            'Each bar shows how many scenarios produced that price. The green line is our base target (median outcome).',
            ha='center', fontsize=10, style='italic'
        )
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"   ✓ Saved price distribution: {output_path}")
        return output_path


if __name__ == "__main__":
    print("Price Targets Module Ready")

In [ ]:
"""
Risk Analysis Module - Quantifies downside scenarios
Calculates VaR, CVaR, and stress tests
"""

import numpy as np
import pandas as pd


class RiskAnalysis:
    """Measures investment risk in clear terms"""
    
    def __init__(self, ticker, current_price, simulated_prices, expected_return, volatility):
        """
        Args:
            ticker: Your target stock
            current_price: Today's price
            simulated_prices: Array of Monte Carlo outcomes
            expected_return: Expected annual return
            volatility: Annual volatility
        """
        self.ticker = ticker
        self.current_price = current_price
        self.simulated_prices = simulated_prices
        self.expected_return = expected_return
        self.volatility = volatility
        
        self.simulated_returns = np.log(simulated_prices / current_price)
    
    def calculate_metrics(self):
        """Calculate all key risk metrics"""
        print(f"\n Analyzing risk for {self.ticker}...")
        
        var_return = np.percentile(self.simulated_returns, 5)
        var_pct = var_return * 100
        
        worst_5pct = self.simulated_returns[self.simulated_returns <= var_return]
        cvar_return = np.mean(worst_5pct)
        cvar_pct = cvar_return * 100
        
        sharpe = self.expected_return / self.volatility if self.volatility > 0 else 0
        
        max_dd = np.min(self.simulated_returns) * 100
        
        stressed_volatility = self.volatility * 1.5
        stressed_sharpe = self.expected_return / stressed_volatility if stressed_volatility > 0 else 0
        
        self.metrics = {
            'var_5pct': var_pct,
            'cvar_5pct': cvar_pct,
            'max_drawdown': max_dd,
            'sharpe_ratio': sharpe,
            'stressed_sharpe': stressed_sharpe
        }
        
        print(f"   ✓ 5% VaR: {var_pct:.1f}% (1-in-20 bad year loss)")
        print(f"   ✓ 5% CVaR: {cvar_pct:.1f}% (average loss in worst scenarios)")
        print(f"   ✓ Sharpe Ratio: {sharpe:.2f} (return per unit risk)")
        print(f"   ✓ Max Drawdown: {max_dd:.1f}%")
        print(f"   ✓ Crash-Stressed Sharpe: {stressed_sharpe:.2f}")
        
        return self.metrics
    
    def create_table(self):
        """Generate risk metrics table for your pitch deck"""
        table = pd.DataFrame({
            'Risk Metric': [
                '5% VaR (1-in-20 bad year)',
                '5% CVaR (average in worst 5%)',
                'Max Drawdown',
                'Sharpe Ratio (Base)',
                'Sharpe Ratio (Market Crash)'
            ],
            'Value': [
                f"{self.metrics['var_5pct']:.1f}%",
                f"{self.metrics['cvar_5pct']:.1f}%",
                f"{self.metrics['max_drawdown']:.1f}%",
                f"{self.metrics['sharpe_ratio']:.2f}",
                f"{self.metrics['stressed_sharpe']:.2f}"
            ],
            'Explanation': [
                'Expected loss in a bad 1-in-20 year',
                'If we hit that bad scenario, average loss',
                'Worst-case drop from peak in simulations',
                'Return we earn per unit of volatility taken',
                'Sharpe if market volatility increases 50%'
            ]
        })
        
        return table


if __name__ == "__main__":
    print("Risk Analysis Module Ready")

In [ ]:
"""
Impact Scoring Module - Quantifies ESG and AI benefits
Simple scoring for environmental and growth factors
"""

import pandas as pd


class ImpactScoring:
    """Scores ESG quality and AI growth impact"""
    
    def __init__(self, ticker):
        self.ticker = ticker
        self.esg_score = None
        self.ai_uplift = None
    
    def score(self, esg_score=9.0, ai_revenue_uplift=0.20):
        """
        Assign scores and calculate adjusted returns
        
        Args:
            esg_score: Environmental score 0-10 (nuclear = 9)
            ai_revenue_uplift: Revenue boost from AI datacenters (20% = 0.20)
        """
        print(f"\n Scoring ESG & AI impact for {self.ticker}...")
        
        self.esg_score = esg_score
        self.ai_uplift = ai_revenue_uplift
        
        if esg_score >= 8:
            esg_reason = "Zero-carbon baseload power (nuclear SMR)"
        elif esg_score >= 6:
            esg_reason = "Low carbon footprint"
        else:
            esg_reason = "Moderate environmental profile"
        
        ai_reason = f"AI datacenter demand drives {ai_revenue_uplift*100:.0f}% revenue premium"
        
        print(f"   ✓ ESG Score: {esg_score}/10 - {esg_reason}")
        print(f"   ✓ AI Impact: +{ai_revenue_uplift*100:.0f}% - {ai_reason}")
        
        self.explanations = {'esg': esg_reason, 'ai': ai_reason}
        
        return {
            'esg_score': esg_score,
            'ai_uplift': ai_revenue_uplift,
            'esg_reason': esg_reason,
            'ai_reason': ai_reason
        }
    
    def adjust_return(self, base_return):
        """Adjust expected return for ESG/AI factors"""
        esg_premium = max(0, (self.esg_score - 5.0) * 0.005)
        ai_premium = self.ai_uplift
        
        adjusted = base_return + esg_premium + ai_premium
        
        print(f"   ✓ Base return: {base_return:.1%}")
        print(f"   ✓ ESG premium: +{esg_premium:.1%}")
        print(f"   ✓ AI premium: +{ai_premium:.1%}")
        print(f"   ✓ Adjusted return: {adjusted:.1%}")
        
        return adjusted
    
    def create_summary_table(self):
        """Create summary table for pitch deck"""
        table = pd.DataFrame({
            'Factor': ['ESG Score', 'AI Revenue Uplift'],
            'Value': [f'{self.esg_score}/10', f'+{self.ai_uplift*100:.0f}%'],
            'Why It Matters': [
                self.explanations['esg'],
                self.explanations['ai']
            ]
        })
        return table


if __name__ == "__main__":
    print("Impact Scoring Module Ready")

In [ ]:
"""
HANGAR-YIS Pitch Edition
Simple quantitative analysis for stock pitch competitions
"""

import os
from datetime import datetime
import json


def run_analysis(ticker, peers, ai_uplift=0.20, esg_score=9.0, output_dir='outputs'):
    print("="*70)
    print("HANGAR-YIS PITCH EDITION")
    print("Quantitative Stock Analysis Tool")
    print("="*70)
    print(f"\n Target: {ticker}")
    print(f" Peers: {', '.join(peers)}")
    print(f" AI Uplift: +{ai_uplift*100:.0f}%  |  ESG Score: {esg_score}/10")
    
    os.makedirs(output_dir, exist_ok=True)
    all_tickers = [ticker] + peers
    
    # STEP 1: Data
    print("\n" + "="*70)
    print("STEP 1: FETCHING DATA")
    print("="*70)
    
    fetcher = DataFetcher(ticker, peers)
    fetcher.fetch()
    mean_returns, cov_matrix = fetcher.calculate_statistics()
    current_price = fetcher.current_prices[ticker]
    
    # STEP 2: Peers
    print("\n" + "="*70)
    print("STEP 2: PEER ANALYSIS")
    print("="*70)
    
    peer_comp = PeerAnalysis(ticker, all_tickers, cov_matrix)
    peer_table = peer_comp.analyze()
    peer_comp.plot_similarity_map(f'{output_dir}/peer_similarity_map.png')
    peer_table.to_markdown(f'{output_dir}/peer_analysis.md', index=False)
    
    # STEP 3: ESG / AI
    print("\n" + "="*70)
    print("STEP 3: ESG & AI IMPACT")
    print("="*70)
    
    impact = ImpactScoring(ticker)
    impact.score(esg_score=esg_score, ai_revenue_uplift=ai_uplift)
    base_return = mean_returns.iloc[0] if hasattr(mean_returns, 'iloc') else mean_returns[0]
    adjusted_return = impact.adjust_return(base_return)
    
    # Create a copy of mean_returns for modification
    mean_returns_arr = mean_returns.values.copy() if hasattr(mean_returns, 'values') else mean_returns.copy()
    mean_returns_arr[0] = adjusted_return
    
    impact_table = impact.create_summary_table()
    impact_table.to_markdown(f'{output_dir}/esg_ai_impact.md', index=False)
    
    # STEP 4: Price Targets
    print("\n" + "="*70)
    print("STEP 4: PRICE TARGETS (Monte Carlo)")
    print("="*70)
    
    targets = PriceTargets(
        ticker,
        current_price,
        adjusted_return,
        fetcher.volatility[0],
        cov_matrix,
        mean_returns_arr
    )
    
    target_results = targets.simulate(n_sims=10000, seed=42)
    targets.plot_distribution(
        f'{output_dir}/price_targets.png',
        analyst_targets=[140, 180]
    )
    
    # STEP 5: Risk
    print("\n" + "="*70)
    print("STEP 5: RISK METRICS")
    print("="*70)
    
    risk = RiskAnalysis(
        ticker,
        current_price,
        targets.simulated_prices,
        adjusted_return,
        fetcher.volatility[0]
    )
    
    risk_metrics = risk.calculate_metrics()
    risk_table = risk.create_table()
    risk_table.to_markdown(f'{output_dir}/risk_metrics.md', index=False)
    
    # SUMMARY
    print("\n" + "="*70)
    print(" ANALYSIS COMPLETE - SUMMARY")
    print("="*70)
    
    upside_pct = (target_results['base'] / current_price - 1) * 100
    
    summary = f"""
{ticker} INVESTMENT THESIS

Current Price:        ${current_price:.2f}
Base Target (1Y):     ${target_results['base']:.2f} (+{upside_pct:.1f}%)
Upside Target (75th): ${target_results['upside']:.2f}
Downside (25th):      ${target_results['downside']:.2f}

Risk-Adjusted Return: Sharpe Ratio {risk_metrics['sharpe_ratio']:.2f}
Downside Protection:  5% CVaR {risk_metrics['cvar_5pct']:.1f}%

ESG Score:           {esg_score}/10 - {impact.explanations['esg']}
AI Growth Driver:    +{ai_uplift*100:.0f}% - {impact.explanations['ai']}

INVESTMENT CASE:
{ticker} offers {upside_pct:.0f}% upside to our base target,
driven by structural AI datacenter demand for zero-carbon baseload power.
Risk-adjusted returns (Sharpe {risk_metrics['sharpe_ratio']:.2f}) support the thesis,
with downside framed through VaR/CVaR rather than pure story.
"""
    print(summary)
    
    with open(f'{output_dir}/investment_thesis.txt', 'w') as f:
        f.write(summary)
    
    full_results = {
        'ticker': ticker,
        'analysis_date': datetime.now().isoformat(),
        'current_price': float(current_price),
        'targets': {
            'base': float(target_results['base']),
            'upside': float(target_results['upside']),
            'downside': float(target_results['downside']),
            'upside_pct': float(upside_pct)
        },
        'risk': {
            'sharpe': float(risk_metrics['sharpe_ratio']),
            'var_5pct': float(risk_metrics['var_5pct']),
            'cvar_5pct': float(risk_metrics['cvar_5pct'])
        },
        'impact': {
            'esg_score': float(esg_score),
            'ai_uplift_pct': float(ai_uplift * 100)
        },
        'peers': peer_table.to_dict('records')
    }
    
    with open(f'{output_dir}/results.json', 'w') as f:
        json.dump(full_results, f, indent=2)
    
    print(f"\n📁 All outputs saved to: {output_dir}/")
    print("   ✓ peer_similarity_map.png")
    print("   ✓ price_targets.png")
    print("   ✓ peer_analysis.md")
    print("   ✓ risk_metrics.md")
    print("   ✓ esg_ai_impact.md")
    print("   ✓ investment_thesis.txt")
    print("   ✓ results.json")
    
    return full_results


# Run the analysis
run_analysis(
    ticker='OKLO',
    peers=['SMR', 'BWXT', 'CCJ'],
    ai_uplift=0.20,
    esg_score=9.0,
    output_dir='outputs'
)

# Math Behind the Decisions

This notebook makes quantitative decisions using a small set of consistent equations.

1. Returns and risk inputs
Daily returns are `r_t = (P_t / P_{t-1}) - 1`. Annualized expected return is `mu = mean(r) * 252` and annualized covariance is `Sigma = cov(r) * 252`. Volatility is `sigma = sqrt(diag(Sigma))`.

2. Peer similarity and clustering
Correlation is `rho_ij = Sigma_ij / (sigma_i * sigma_j)`. The distance used for clustering is `d_ij = sqrt(Sigma_ii + Sigma_jj - 2 * Sigma_ij)` and single-linkage hierarchical clustering groups the closest movers.

3. ESG and AI adjustments
Expected return is adjusted as `mu_adj = mu_base + max(0, (ESG - 5) * 0.005) + AI_uplift` to reflect ESG quality and AI-driven revenue uplift.

4. Price targets (Monte Carlo)
Simulated annual returns follow `R ~ N(mu_adj, Sigma)`. For the target stock, simulated prices are `P_1 = P_0 * exp(R)`. Downside, base, and upside targets are the 25th, 50th, and 75th percentiles of simulated prices.

5. Risk metrics
Simulated log returns are `log(P_1 / P_0)`. VaR 5% is the 5th percentile of these returns. CVaR 5% is the mean of the worst 5%. Sharpe is `mu_adj / sigma`. The stress test increases volatility by 50% to compute a crash-stressed Sharpe. The minimum simulated log return is used as a max drawdown proxy.
